# Session 2 - The Model That Cheated

## MISSION

> Four candidate models will be built under a frozen time split. One candidate will be deliberately contaminated with future information, then it will be exposed and excluded before a legitimate champion is selected.

### Success criteria
- Four candidates will be evaluated through one shared metric contract.
- The higher-scoring leaky candidate will be rejected before ranking.
- The strongest eligible model will be selected by Macro F1, then log loss.
- Three auditable artifacts will be exported.

**Primary artifact:** Leakage-Aware Model Validator


## MISSION MAP

`FUSED DATA -> FREEZE TIME -> DEFINE THE METRIC -> TRAIN FAIR MODELS -> BUILD THE CHEATING MODEL -> EXCLUDE -> SELECT`


## OPENING CHALLENGE

| Candidate | Macro F1 | Prediction-time valid? | Decision |
|---|---:|---|---|
| Logistic | 0.6357 | Yes | Eligible |
| Leaky XGBoost | 0.9654 | No - `future_loss_index` is post-outcome | Disqualified |

> The higher score will be rejected because eligibility is decided before performance is ranked.


## F1 AND MACRO F1 - THE 90-SECOND FOUNDATION

**Precision** asks: of everything predicted as a class, how much was correct?  
**Recall** asks: of all real members of that class, how much was found?

$$F1 = \frac{2 \times Precision \times Recall}{Precision + Recall}$$

The harmonic mean is used, so a very weak precision or recall value cannot be hidden by a strong value on the other side.


### Why Macro F1 is used here

$$Macro\ F1 = \frac{F1_{LOW} + F1_{MEDIUM} + F1_{HIGH}}{3}$$

Every risk class receives equal influence, so strong LOW-risk performance cannot hide failure on the smaller HIGH-risk class. However, Macro F1 does **not** detect leakage; feature eligibility and evaluation design must be validated first.


## 1. Path setup and imports

In [1]:
# The system module is imported so Python's module search path can be updated.
import sys
# Path is imported so filesystem locations can be handled portably.
from pathlib import Path

# Find project root by searching upward for src/
# The current working directory is captured as the starting location.
current = Path.cwd()
# Each item in the specified collection is processed in sequence.
for parent in [current] + list(current.parents):
    # The required condition is checked before this branch is executed.
    if (parent / "src").exists():
        # The detected project root is stored for subsequent path construction.
        root = parent
        # The search loop is stopped after the required project root is found.
        break
# The alternative branch is entered when the preceding condition is not satisfied.
else:
    # The detected project root is stored for subsequent path construction.
    root = current

# The required condition is checked before this branch is executed.
if str(root / "src") not in sys.path:
    # The project source directory is added to the front of Python's import path.
    sys.path.insert(0, str(root / "src"))

# The canonical data directory is defined relative to the project root.
DATA_DIR = root / "src" / "data"
# The canonical report directory is defined relative to the project root.
REPORTS_DIR = root / "src" / "reports"
# The required output directory is created when it does not already exist.
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {root}")

print(f"Data dir: {DATA_DIR}")

print(f"Reports dir: {REPORTS_DIR}")


Project root: c:\Users\Metwalli\Desktop\summer\Advanced_AI_Practical
Data dir: c:\Users\Metwalli\Desktop\summer\Advanced_AI_Practical\src\data
Reports dir: c:\Users\Metwalli\Desktop\summer\Advanced_AI_Practical\src\reports


In [ ]:
# NumPy is imported for arrays, class labels, and numerical operations.
import numpy as np
# Pandas is imported for tabular data loading, filtering, and export.
import pandas as pd

# A fixed random seed is declared so repeated runs remain reproducible.
RANDOM_STATE = 42
# The composite observation key is declared for region-month records.
KEYS = ["region_id", "year", "month"]
# The ordered multiclass risk labels are declared.
CLASS_NAMES = np.array(["LOW", "MEDIUM", "HIGH"])
# A deterministic mapping from class names to integer IDs is created.
LABEL_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}

# The approved raw measurement features are started as an explicit registry.
RAW_FEATURES = [
    
    "temperature_c", "rainfall_mm", "humidity_pct",
    
    "soil_moisture", "soil_ph", "vegetation_index",
    
    "disease_cases", "pest_reports",
    
    "irrigation_capacity", "vulnerability_index",
# The current list or pipeline definition is closed.
]

# The derived modeling features are started as an explicit registry.
ENGINEERED_FEATURES = [
    
    "drought_index", "heat_stress", "disease_pressure",
    
    "water_balance", "season_sin", "season_cos",
# The current list or pipeline definition is closed.
]

# The complete legitimate feature set is assembled from raw and engineered features.
MODEL_FEATURES = RAW_FEATURES + ENGINEERED_FEATURES
# Post-outcome features are explicitly quarantined from legitimate modeling.
PROHIBITED_FEATURES = ["future_loss_index", "response_action_code"]


print("Constants defined")

print(f"Model features: {len(MODEL_FEATURES)}")

print(f"Prohibited features: {PROHIBITED_FEATURES}")


Constants defined
Model features: 16
Prohibited features: ['future_loss_index', 'response_action_code']


## 2. Load fused data from processed directory

In [ ]:
# The processed-data directory is resolved.
processed_dir = DATA_DIR / "processed"
# The expected fused modeling-table path is constructed.
fused_path = processed_dir / "modeling_table.csv"

# The required condition is checked before this branch is executed.
if not fused_path.exists():
    
    print("Processed data not found. Generating from scratch...")
    # Required components are imported from reliable_ai.data.
    from reliable_ai.data import fuse_sources
    # The value required for fused is assigned.
    fused = fuse_sources(DATA_DIR)
    # The required output directory is created when it does not already exist.
    processed_dir.mkdir(parents=True, exist_ok=True)
    # The resulting table is exported as a reproducible CSV artifact.
    fused.to_csv(fused_path, index=False)
# The alternative branch is entered when the preceding condition is not satisfied.
else:
    # The previously prepared modeling table is loaded from disk.
    fused = pd.read_csv(fused_path)


print(f"Fused data: {fused.shape[0]} rows x {fused.shape[1]} columns")


Fused data: 840 rows x 24 columns


## 3. RUN AND INSPECT - Explore the fused data

In [ ]:

print(f"Year range: {fused['year'].min()} - {fused['year'].max()}")

print(f"Regions: {fused['region_id'].nunique()}")

print(f"\nRisk label distribution:")

print(fused["risk_label"].value_counts().to_string())

print(f"\nProhibited features present: {[c for c in PROHIBITED_FEATURES if c in fused.columns]}")


Year range: 2019 - 2025
Regions: 10

Risk label distribution:
risk_label
LOW       471
MEDIUM    347
HIGH       22

Prohibited features present: ['future_loss_index', 'response_action_code']


## 4. PREDICT BEFORE RUNNING

> If a model is given access to `future_loss_index` (a post-outcome proxy), how will its macro F1 compare to a model that only uses legitimate features?

Record your hypothesis:

*Your hypothesis:* The leaky model will score higher because it has direct access to information correlated with the target that would not exist at prediction time.

## 5. GUIDED BUILD - Chronological split and future holdout

In [ ]:
# Chronological split: train on past, calibrate on 2024, test on 2025
# An independent dataframe copy is created to prevent accidental chained modification.
train = fused[fused["year"] <= 2023].copy()
# An independent dataframe copy is created to prevent accidental chained modification.
calibration = fused[fused["year"] == 2024].copy()
# An independent dataframe copy is created to prevent accidental chained modification.
test = fused[fused["year"] == 2025].copy()


print(f"Train: {len(train)} rows ({train['year'].min()}-{train['year'].max()})")

print(f"Calibration: {len(calibration)} rows ({calibration['year'].min()})")

print(f"Test: {len(test)} rows ({test['year'].min()})")


Train: 600 rows (2019-2023)
Calibration: 120 rows (2024)
Test: 120 rows (2025)


In [ ]:
# Assertion checkpoint: all partitions non-empty
# A defensive assertion is evaluated so an invalid state is stopped immediately.
assert len(train) > 0, "Train partition is empty"
# A defensive assertion is evaluated so an invalid state is stopped immediately.
assert len(calibration) > 0, "Calibration partition is empty"
# A defensive assertion is evaluated so an invalid state is stopped immediately.
assert len(test) > 0, "Test partition is empty"

print("Chronological split verified - all partitions populated")


Chronological split verified - all partitions populated


## 6. LAB ACTION - Encode target and build candidates

In [ ]:
# Encode target labels: LOW=0, MEDIUM=1, HIGH=2
# A reusable helper named encode_target is defined.
def encode_target(y):
    
    """Map string risk labels to integer class IDs."""
    # Human-readable risk labels are mapped to integer class identifiers.
    return y.map(LABEL_TO_ID).values.astype(int)

# Training targets are encoded as integer class IDs.
y_train = encode_target(train["risk_label"])
# Holdout targets are encoded as integer class IDs.
y_test = encode_target(test["risk_label"])

print(f"Encoded train classes: {np.unique(y_train)}")

print(f"Encoded test classes: {np.unique(y_test)}")


Encoded train classes: [0 1 2]
Encoded test classes: [0 1 2]


In [ ]:
# Only legitimate model features are selected for training.
X_train = train[MODEL_FEATURES]
# The same legitimate feature contract is applied to the holdout data.
X_test = test[MODEL_FEATURES]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")


X_train: (600, 16), X_test: (120, 16)


## 7. Build candidate models

In [ ]:
# Required components are imported from sklearn.dummy.
from sklearn.dummy import DummyClassifier
# Required components are imported from sklearn.impute.
from sklearn.impute import SimpleImputer
# Required components are imported from sklearn.preprocessing.
from sklearn.preprocessing import StandardScaler
# Required components are imported from sklearn.linear_model.
from sklearn.linear_model import LogisticRegression
# Required components are imported from sklearn.compose.
from sklearn.compose import ColumnTransformer
# Required components are imported from sklearn.pipeline.
from sklearn.pipeline import Pipeline
# Required components are imported from sklearn.metrics.
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss


print("Imports ready")


Imports ready


In [ ]:
# Candidate 1: Dummy (most_frequent baseline)
# A trivial majority-class baseline is instantiated as the performance floor.
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
# The estimator is fitted using the specified training inputs and labels.
dummy.fit(X_train, y_train)

print("Dummy trained")


Dummy trained


In [ ]:
# Candidate 2: Logistic pipeline (SimpleImputer + StandardScaler + LogisticRegression)
# A leakage-safe processing pipeline is started.
numeric_transformer = Pipeline([
    # Missing numerical values are configured to be replaced by training-set medians.
    ("imputer", SimpleImputer(strategy="median")),
    # Numerical features are configured to be standardized inside the training pipeline.
    ("scaler", StandardScaler()),
# The current nested pipeline or transformer definition is closed.
])
# A column-aware preprocessing transformer is started.
preprocessor = ColumnTransformer([
    # A nested configuration block is opened.
    ("num", numeric_transformer, MODEL_FEATURES),
# The current nested pipeline or transformer definition is closed.
])
# A leakage-safe processing pipeline is started.
logistic = Pipeline([
    # A nested configuration block is opened.
    ("preprocessor", preprocessor),
    # A balanced logistic-regression classifier is configured as the serious linear baseline.
    ("classifier", LogisticRegression(
        # The value required for max_iter is assigned.
        max_iter=2000,
        # The value required for class_weight is assigned.
        class_weight="balanced",
        # The value required for random_state is assigned.
        random_state=RANDOM_STATE,
    # This line advances the current modeling or validation step.
    )),
# The current nested pipeline or transformer definition is closed.
])
# The estimator is fitted using the specified training inputs and labels.
logistic.fit(X_train, y_train)

print("Logistic pipeline trained")


Logistic pipeline trained


In [ ]:
# Candidate 3: Advanced correct (XGBoost with approved features only)
# The preferred implementation is attempted inside a protected block.
try:
    # Required components are imported from xgboost.
    from xgboost import XGBClassifier
    # An XGBoost multiclass classifier is configured as the nonlinear candidate.
    clf = XGBClassifier(
        # The value required for n_estimators is assigned.
        n_estimators=260, max_depth=4, learning_rate=0.055,
        # The value required for subsample is assigned.
        subsample=0.85, colsample_bytree=0.85,
        # The value required for objective is assigned.
        objective="multi:softprob", eval_metric="mlogloss",
        # The value required for random_state is assigned.
        random_state=RANDOM_STATE, n_jobs=1, device="cpu",
    # The current function or constructor call is closed.
    )
    
    print("Using XGBClassifier")
# A compatible fallback is activated when the preferred package is unavailable.
except ImportError:
    # Required components are imported from sklearn.ensemble.
    from sklearn.ensemble import HistGradientBoostingClassifier
    # A histogram gradient-boosting fallback is configured for environments without XGBoost.
    clf = HistGradientBoostingClassifier(
        # The value required for learning_rate is assigned.
        learning_rate=0.07, max_iter=220,
        # The value required for max_leaf_nodes is assigned.
        max_leaf_nodes=15, l2_regularization=0.15,
        # The value required for random_state is assigned.
        random_state=RANDOM_STATE,
    # The current function or constructor call is closed.
    )
    
    print("Using HistGradientBoostingClassifier (XGBoost not available)")

# A column-aware preprocessing transformer is started.
advanced_preprocessor = ColumnTransformer([
    # A leakage-safe processing pipeline is started.
    ("num", Pipeline([
        # Missing numerical values are configured to be replaced by training-set medians.
        ("imputer", SimpleImputer(strategy="median")),
        # Numerical features are configured to be standardized inside the training pipeline.
        ("scaler", StandardScaler()),
    # This line advances the current modeling or validation step.
    ]), MODEL_FEATURES),
# The current nested pipeline or transformer definition is closed.
])
# A leakage-safe processing pipeline is started.
advanced_correct = Pipeline([
    # A nested configuration block is opened.
    ("preprocessor", advanced_preprocessor),
    # A nested configuration block is opened.
    ("classifier", clf),
# The current nested pipeline or transformer definition is closed.
])
# The estimator is fitted using the specified training inputs and labels.
advanced_correct.fit(X_train, y_train)

print("Advanced correct trained")


Using XGBClassifier
Advanced correct trained


In [ ]:
# Candidate 4: Advanced leaky (same XGBoost but with future_loss_index)
# The prohibited future feature is deliberately added for the controlled leakage experiment.
leaky_features = MODEL_FEATURES + ["future_loss_index"]

# A column-aware preprocessing transformer is started.
leaky_preprocessor = ColumnTransformer([
    # A leakage-safe processing pipeline is started.
    ("num", Pipeline([
        # Missing numerical values are configured to be replaced by training-set medians.
        ("imputer", SimpleImputer(strategy="median")),
        # Numerical features are configured to be standardized inside the training pipeline.
        ("scaler", StandardScaler()),
    # This line advances the current modeling or validation step.
    ]), leaky_features),
# The current nested pipeline or transformer definition is closed.
])

# The preferred implementation is attempted inside a protected block.
try:
    # Required components are imported from xgboost.
    from xgboost import XGBClassifier
    # An XGBoost multiclass classifier is configured as the nonlinear candidate.
    leaky_clf = XGBClassifier(
        # The value required for n_estimators is assigned.
        n_estimators=260, max_depth=4, learning_rate=0.055,
        # The value required for subsample is assigned.
        subsample=0.85, colsample_bytree=0.85,
        # The value required for objective is assigned.
        objective="multi:softprob", eval_metric="mlogloss",
        # The value required for random_state is assigned.
        random_state=RANDOM_STATE, n_jobs=1, device="cpu",
    # The current function or constructor call is closed.
    )
# A compatible fallback is activated when the preferred package is unavailable.
except ImportError:
    # Required components are imported from sklearn.ensemble.
    from sklearn.ensemble import HistGradientBoostingClassifier
    # A histogram gradient-boosting fallback is configured for environments without XGBoost.
    leaky_clf = HistGradientBoostingClassifier(
        # The value required for learning_rate is assigned.
        learning_rate=0.07, max_iter=220,
        # The value required for max_leaf_nodes is assigned.
        max_leaf_nodes=15, l2_regularization=0.15,
        # The value required for random_state is assigned.
        random_state=RANDOM_STATE,
    # The current function or constructor call is closed.
    )

# A leakage-safe processing pipeline is started.
leaky_pipeline = Pipeline([
    # A nested configuration block is opened.
    ("preprocessor", leaky_preprocessor),
    # A nested configuration block is opened.
    ("classifier", leaky_clf),
# The current nested pipeline or transformer definition is closed.
])
# The estimator is fitted using the specified training inputs and labels.
leaky_pipeline.fit(train[MODEL_FEATURES + ["future_loss_index"]], y_train)

print("Advanced leaky trained")


Advanced leaky trained


## 8. RUN AND INSPECT - Score all candidates on the future holdout

In [ ]:
# Compute metrics for one candidate: accuracy, balanced accuracy, macro F1, log loss
# A reusable helper named metric_row is defined.
def metric_row(name, model, X, y_true, features):
    # Class labels are predicted for the supplied evaluation records.
    y_pred = model.predict(X[features])
    # Class probabilities are generated for probability-sensitive evaluation.
    proba = model.predict_proba(X[features])
    # The computed metrics are returned as one structured dictionary.
    return {
        
        "model": name,
        # Overall classification accuracy is calculated.
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        # Overall classification accuracy is calculated.
        "balanced_accuracy": round(balanced_accuracy_score(y_true, y_pred), 4),
        # Macro F1 is calculated so every risk class contributes equally.
        "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
        # Log loss is calculated so the quality of predicted probabilities is measured.
        "log_loss": round(log_loss(y_true, proba), 4),
    # The current dictionary or configuration block is closed.
    }


print("metric_row defined")


metric_row defined


In [ ]:
# All candidate estimators and their required feature sets are registered.
models = {
    
    "dummy": (dummy, MODEL_FEATURES),
    
    "logistic": (logistic, MODEL_FEATURES),
    
    "advanced_correct": (advanced_correct, MODEL_FEATURES),
    
    "advanced_leaky": (leaky_pipeline, MODEL_FEATURES + ["future_loss_index"]),
# The current dictionary or configuration block is closed.
}

# An empty collection is prepared for candidate metric records.
rows = []
# Each item in the specified collection is processed in sequence.
for name, (model, feats) in models.items():
    # One candidate metric record is computed.
    row = metric_row(name, model, test, y_test, feats)
    # The current candidate record is appended to the comparison collection.
    rows.append(row)

# A structured dataframe is created from the accumulated records.
metrics = pd.DataFrame(rows)

print("All candidates scored")


All candidates scored


In [ ]:
# Sort ALL candidates by macro_f1 descending
# The table is ordered according to the declared selection evidence.
all_sorted = metrics.sort_values("macro_f1", ascending=False).reset_index(drop=True)

print("=== ALL CANDIDATES (sorted by macro F1) ===")

print(all_sorted.to_string(index=False))


=== ALL CANDIDATES (sorted by macro F1) ===
           model  accuracy  balanced_accuracy  macro_f1  log_loss
  advanced_leaky    0.9500             0.9706    0.9654    0.2316
        logistic    0.7417             0.7418    0.6357    0.5700
advanced_correct    0.7000             0.5030    0.4703    0.9523
           dummy    0.4083             0.3333    0.1933   21.3258


## 9. CHECKPOINT - Prediction checkpoint

> Before looking at the metrics, predict which model will have the highest macro F1 and which will be selected after eligibility filtering.

*Your prediction:* The leaky model will have the highest macro F1. After filtering, logistic will be selected.

## 10. BREAK IT - Leaky model wins the raw ranking

In [ ]:

print("=== RAW RANKING (ALL candidates) ===")

print(metrics.sort_values("macro_f1", ascending=False).to_string(index=False))

# The deliberately leaky candidate record is isolated for reporting.
leaky_row = metrics[metrics["model"] == "advanced_leaky"].iloc[0]

print(f"\nLeaky model macro F1: {leaky_row['macro_f1']:.4f}")

print("If we selected by score alone, the leaky model would win.")


=== RAW RANKING (ALL candidates) ===
           model  accuracy  balanced_accuracy  macro_f1  log_loss
  advanced_leaky    0.9500             0.9706    0.9654    0.2316
        logistic    0.7417             0.7418    0.6357    0.5700
advanced_correct    0.7000             0.5030    0.4703    0.9523
           dummy    0.4083             0.3333    0.1933   21.3258

Leaky model macro F1: 0.9654
If we selected by score alone, the leaky model would win.


## 10. BREAK IT - Controlled failure: selecting without eligibility

In [ ]:
# This is what happens when you ignore feature eligibility
# The table is ordered according to the declared selection evidence.
naive_best = metrics.sort_values("macro_f1", ascending=False).iloc[0]

print(f"Naive selection would pick: {naive_best['model']}")

print(f"With macro F1: {naive_best['macro_f1']:.4f}")

print("But this model uses future_loss_index - a post-outcome proxy!")

print("It would be impossible to use at prediction time.")


Naive selection would pick: advanced_leaky
With macro F1: 0.9654
But this model uses future_loss_index - a post-outcome proxy!
It would be impossible to use at prediction time.


## 11. FIX IT - Apply eligibility filter and re-rank

In [ ]:
# Filter to only legitimate models (exclude advanced_leaky)
# The approved candidate names are declared before ranking.
LEGITIMATE_MODEL_NAMES = ["dummy", "logistic", "advanced_correct"]

# An independent dataframe copy is created to prevent accidental chained modification.
legitimate = metrics[metrics["model"].isin(LEGITIMATE_MODEL_NAMES)].copy()

print(f"Legitimate candidates: {legitimate['model'].tolist()}")


Legitimate candidates: ['dummy', 'logistic', 'advanced_correct']


In [ ]:
# Re-rank legitimate models: highest macro_f1, then lowest log_loss
# The table is ordered according to the declared selection evidence.
sorted_legit = legitimate.sort_values(
    # The value required for by is assigned.
    by=["macro_f1", "log_loss", "model"],
    # The value required for ascending is assigned.
    ascending=[False, True, True],
# A clean sequential index is restored after sorting or filtering.
).reset_index(drop=True)


print("=== LEGITIMATE CANDIDATES (re-ranked) ===")

print(sorted_legit.to_string(index=False))

# The highest-ranked legitimate candidate is selected.
selected = sorted_legit.iloc[0]

print(f"\nSelected model: {selected['model']}")

print(f"Macro F1: {selected['macro_f1']:.4f}")

print(f"Log loss: {selected['log_loss']:.4f}")


=== LEGITIMATE CANDIDATES (re-ranked) ===
           model  accuracy  balanced_accuracy  macro_f1  log_loss
        logistic    0.7417             0.7418    0.6357    0.5700
advanced_correct    0.7000             0.5030    0.4703    0.9523
           dummy    0.4083             0.3333    0.1933   21.3258

Selected model: logistic
Macro F1: 0.6357
Log loss: 0.5700


In [ ]:
# Assertion checkpoint: logistic must be selected
# A defensive assertion is evaluated so an invalid state is stopped immediately.
assert selected["model"] == "logistic", f"Expected logistic, got {selected['model']}"

print("Assertion passed: logistic selected as best legitimate model")


Assertion passed: logistic selected as best legitimate model


## 12. MINI-PROJECT - Export artifacts

In [ ]:
# Export model comparison
# The resulting table is exported as a reproducible CSV artifact.
metrics.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)

print("Exported: model_comparison.csv")


Exported: model_comparison.csv


In [ ]:
# Build and export feature availability audit
# An empty collection is prepared for the feature-availability audit.
audit_rows = []
# Each item in the specified collection is processed in sequence.
for feat in MODEL_FEATURES:
    # One feature-availability record is appended to the audit collection.
    audit_rows.append({
        
        "feature": feat,
        
        "model_role": "approved",
        
        "prediction_time_availability": "available",
        
        "eligibility": "eligible",
        
        "reason": "current measurement",
    # This line advances the current modeling or validation step.
    })
# Each item in the specified collection is processed in sequence.
for feat in PROHIBITED_FEATURES:
    # One feature-availability record is appended to the audit collection.
    audit_rows.append({
        
        "feature": feat,
        
        "model_role": "prohibited",
        
        "prediction_time_availability": (
            
            "post_outcome_only" if feat == "future_loss_index" else "post_response"
        # This line advances the current modeling or validation step.
        ),
        
        "eligibility": "prohibited",
        
        "reason": f"{feat} is not available at prediction time",
    # This line advances the current modeling or validation step.
    })

# A structured dataframe is created from the accumulated records.
feature_audit = pd.DataFrame(audit_rows)
# The resulting table is exported as a reproducible CSV artifact.
feature_audit.to_csv(REPORTS_DIR / "feature_availability_audit.csv", index=False)

print(feature_audit.to_string(index=False))


             feature model_role prediction_time_availability eligibility                                                   reason
       temperature_c   approved                    available    eligible                                      current measurement
         rainfall_mm   approved                    available    eligible                                      current measurement
        humidity_pct   approved                    available    eligible                                      current measurement
       soil_moisture   approved                    available    eligible                                      current measurement
             soil_ph   approved                    available    eligible                                      current measurement
    vegetation_index   approved                    available    eligible                                      current measurement
       disease_cases   approved                    available    eligible                  

In [ ]:
# Build and export model selection record
# The selected model record is isolated from the metric table.
selected_row = metrics[metrics["model"] == selected["model"]].iloc[0]
# The deliberately leaky candidate record is isolated for reporting.
leaky_row = metrics[metrics["model"] == "advanced_leaky"].iloc[0]

# A structured dataframe is created from the accumulated records.
selection_record = pd.DataFrame([{
    
    "selected_model_name": selected["model"],
    
    "selected_macro_f1": selected["macro_f1"],
    
    "selection_rule": "highest legitimate macro_f1, then lowest log_loss",
    
    "excluded_candidate": "advanced_leaky",
    
    "excluded_reason": "uses future_loss_index (post-outcome proxy, unavailable at prediction time)",
    
    "train_years": "2019-2023",
    
    "calibration_year": 2024,
    
    "test_year": 2025,
# This line advances the current modeling or validation step.
}])
# The resulting table is exported as a reproducible CSV artifact.
selection_record.to_csv(REPORTS_DIR / "model_selection_record.csv", index=False)

print(selection_record.to_string(index=False))


selected_model_name  selected_macro_f1                                    selection_rule excluded_candidate                                                             excluded_reason train_years  calibration_year  test_year
           logistic             0.6357 highest legitimate macro_f1, then lowest log_loss     advanced_leaky uses future_loss_index (post-outcome proxy, unavailable at prediction time)   2019-2023              2024       2025


## 12. EVIDENCE - Leakage-Aware Model Validator

In [ ]:

print("=" * 60)

print("LEAKAGE-AWARE MODEL VALIDATOR")

print("=" * 60)

print(f"{'Total candidates':30s}: 4")

print(f"{'Eligible candidates':30s}: 3")

print(f"{'Excluded (leaky)':30s}: advanced_leaky")

print(f"{'Leaky macro F1':30s}: {leaky_row['macro_f1']:.4f}")

print(f"{'Selected model':30s}: {selected['model']}")

print(f"{'Selected macro F1':30s}: {selected['macro_f1']:.4f}")

print(f"{'Selection rule':30s}: highest legitimate macro_f1, then lowest log_loss")

print(f"{'Validation status':30s}: PASSED")


LEAKAGE-AWARE MODEL VALIDATOR
Total candidates              : 4
Eligible candidates           : 3
Excluded (leaky)              : advanced_leaky
Leaky macro F1                : 0.9654
Selected model                : logistic
Selected macro F1             : 0.6357
Selection rule                : highest legitimate macro_f1, then lowest log_loss
Validation status             : PASSED


## 13. ASSIGNMENT

**Assignment 02 - Leakage Detection** extends the same pattern to a new set of candidate models with hidden leakage paths.

See `assignments/Assignment_02_Leakage_Detection.pdf`.

## SESSION RESULT

> A model with the strongest legitimate later-year evidence has been identified. A falsely impressive model was created and caught.

The next question becomes: Can the selected model be trusted to make calibrated, reliable decisions under uncertainty?

## TECHNICAL BASIS

- F1 and Macro F1 follow the official scikit-learn metric definitions.
- Leakage controls follow scikit-learn guidance on prediction-time information and pipeline-contained preprocessing.
- The chronological design follows the principle that later observations should be used to evaluate future generalization.
